# Drop seizures: how much they fell, and how that relates to outcome and seizure type

Drop seizures (sudden falls) are the main reason for a callosotomy. `drop_sz_reduction_pct` is the percent change in drop seizures at 6 months, from the post-op neurology note: **-100 = no drop seizures left**, 0 = no change, and nobody got worse. `drop_free` is 1 for -100 and 0 otherwise (`cc_stats/build_analysis.py`). Here the change is shown as a **reduction** (100% = drop-free), so higher is better.

- **Only some patients have it.** It is recorded for about 4 in 10 patients, and a blank could mean no drop seizures before surgery or simply not recorded; that is a question for the data owner. Section 2 compares who has it with who doesn't.
- **It is recorded in steps** (100%, 95%, 90%, 75%, 50%, 25%, 0%), so the values are treated as ordered categories: drop-free yes / no, ranks, and medians, never means.
- **It overlaps the outcome scales.** Engel and ILAE measure seizure reduction, drop seizures included, so drop-seizure change and outcome class go together partly by construction.

Seizure types are the five core groups (atonic, absence, BTC, tonic, myoclonic) and LGS. Every outcome comparison is shown for Engel then ILAE (see `cc_stats/outcomes.py`).

`drop_seizures.py` computes the tables, and this notebook runs it. The output goes to `results/<version>/3_drop_seizures/`.

In [ ]:
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from cc_stats.outcomes import OUTCOMES, heading
from cc_stats.paths import ANALYSIS_CSV, DATASET, PROJECT_ROOT, results_dir
from cc_stats.plots import INK, PAIR_A, PAIR_B, paired_rate_plot, pretty, save_figure

print(f'dataset {DATASET}')
run = subprocess.run([sys.executable, 'analysis/3_drop_seizures/drop_seizures.py'],
                     capture_output=True, text=True, cwd=PROJECT_ROOT)
if run.returncode != 0:
    raise RuntimeError(f'drop_seizures.py failed:\n{run.stderr}')
print(run.stdout.splitlines()[0])

FOLDER = '3_drop_seizures'
RESULTS = results_dir(FOLDER)
analysis = pd.read_csv(ANALYSIS_CSV, index_col='subject_id')
recorded = analysis.dropna(subset=['drop_sz_reduction_pct'])
# Reduction, so higher is better: 100 = drop-free
reduction = -recorded['drop_sz_reduction_pct']
distribution = pd.read_csv(RESULTS / 'drop_distribution.csv')
coverage = pd.read_csv(RESULTS / 'drop_recorded.csv')
by_class = pd.read_csv(RESULTS / 'drop_by_class.csv')
correlation = pd.read_csv(RESULTS / 'drop_class_correlation.csv').set_index('outcome')
by_improved = pd.read_csv(RESULTS / 'drop_by_improvement.csv').set_index('outcome')
by_type = pd.read_csv(RESULTS / 'drop_by_type.csv')


def fmt_p(v):
    return '<0.001' if v < 0.001 else f'{v:.3f}'

## 1. How much drop seizures fell

Patients at each recorded reduction, with the share of all patients who have the value.

In [ ]:
d = distribution.assign(reduction=-distribution['drop_sz_reduction_pct'] + 0)  # + 0: no '-0'.sort_values('reduction', ascending=False)
display(pd.DataFrame({'Reduction': [f'{r:.0f}%' for r in d['reduction']], 'Patients': d['n'],
                      'Share': [f'{s:.0%}' for s in d['share']]}).style.hide(axis='index'))

fig, ax = plt.subplots(figsize=(8, 4))
names = [f'{r:.0f}%' + ('\n(drop-free)' if r == 100 else '\n(no change)' if r == 0 else '') for r in d['reduction']]
bars = ax.bar(names, d['n'], color=PAIR_A)
ax.bar_label(bars, labels=[f'{n}\n{s:.0%}' for n, s in zip(d['n'], d['share'])], padding=2)
ax.set_ylim(0, d['n'].max() * 1.25)
ax.set_xlabel('Reduction in drop seizures at 6 months')
ax.set_ylabel('Patients')
ax.set_title(f'Drop seizures at 6 months (n = {len(recorded)})', loc='left')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
save_figure(FOLDER, 'drop_distribution')
plt.show()

## 2. Who has the value recorded

For each group of patients, how many have drop seizures recorded at 6 months. If the recorded share differs a lot between groups (for example between improved and not improved patients), the results below describe a selected subset.

In [ ]:
display(coverage.assign(**{'Recorded / n': coverage['recorded'].astype(str) + ' / ' + coverage['n'].astype(str),
                           'Share': [f'{s:.0%}' for s in coverage['share']]})[['group', 'Recorded / n', 'Share']]
        .rename(columns={'group': 'Patients'}).style.hide(axis='index'))

## 3. Drop-seizure reduction within each Engel and ILAE class

Every patient's reduction, by outcome class, with the number drop-free in each class above it. Spearman's rho is between the class (higher = worse) and the reduction (higher = better), so a negative rho means worse classes had less reduction.

In [ ]:
for outcome, O in OUTCOMES.items():
    heading(outcome)
    b = by_class[by_class['outcome'] == outcome]
    display(pd.DataFrame({
        'Class': b['class'].map(O['classes']),
        'Drop-free / n': b['drop_free'].astype(str) + ' / ' + b['n'].astype(str),
        'Rate (95% CI)': [f'{r:.0%} ({lo:.0%}-{hi:.0%})' for r, lo, hi in b[['rate', 'lo', 'hi']].to_numpy()],
        'Median reduction': [f'{-m:.0f}%' for m in b['median_change']],
    }).style.hide(axis='index'))

# Both outcomes side by side, each in its colour: every patient as a dot (jittered
# sideways), the class median as a line, and drop-free / n above each class
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True, gridspec_kw={'width_ratios': [4, 6]})
rng = np.random.default_rng(0)
for ax, (outcome, O), color in zip(axes, OUTCOMES.items(), (PAIR_A, PAIR_B)):
    d = recorded.dropna(subset=[O['scale']])
    classes = list(O['classes'])
    for x, c in enumerate(classes):
        values = -d.loc[d[O['scale']] == c, 'drop_sz_reduction_pct']
        if len(values):
            ax.scatter(x + rng.uniform(-0.18, 0.18, len(values)), values, s=28, color=color, alpha=0.75, lw=0)
            ax.hlines(values.median(), x - 0.3, x + 0.3, color=INK, lw=2)
        k = int(d.loc[d[O['scale']] == c, 'drop_free'].sum())
        ax.text(x, 106, f'{k}/{len(values)}', ha='center', va='bottom', fontsize=9, color=INK)
    ax.set_xticks(range(len(classes)), [O['classes'][c] for c in classes])
    ax.set_xlim(-0.6, len(classes) - 0.4)
    ax.set_xlabel(f"{O['name']} class ({O['improved_label'].split(' ', 1)[1]} improved)")
    r = correlation.loc[outcome]
    ax.set_title(f"{O['name']} (n = {int(r['n'])}): rho = {r['rho']:+.2f}, p {fmt_p(r['p']) if r['p'] < 0.001 else '= ' + fmt_p(r['p'])}", loc='left', pad=22)
    ax.grid(axis='y', color='#e5e5e5')
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylim(-5, 115)
axes[0].set_yticks(range(0, 101, 25), [f'{v}%' for v in range(0, 101, 25)])
axes[0].set_ylabel('Reduction in drop seizures (100% = drop-free)')
fig.suptitle('Drop-seizure reduction by outcome class (drop-free / n above each class; line = median)', x=0.01, ha='left')
plt.tight_layout()
save_figure(FOLDER, 'drop_by_class')
plt.show()

## 4. Improved vs not improved

The share drop-free among improved and not-improved patients, for each outcome, with Fisher's exact test. The table also gives the median reduction and a Mann-Whitney test on the whole distribution.

In [ ]:
b = by_improved
display(pd.DataFrame({
    'Outcome': [OUTCOMES[o]['name'] for o in b.index],
    'Improved: drop-free / n': b['drop_free_improved'].astype(str) + ' / ' + b['n_improved'].astype(str),
    'Not improved: drop-free / n': b['drop_free_not_improved'].astype(str) + ' / ' + b['n_not_improved'].astype(str),
    'Median reduction (improved / not)': [f'{-i:.0f}% / {-n:.0f}%' for i, n in b[['median_improved', 'median_not_improved']].to_numpy()],
    "Fisher's p": b['p_fisher'].map(fmt_p),
    'Mann-Whitney p': b['p_mann_whitney'].map(fmt_p),
}).style.hide(axis='index'))

fig, ax = plt.subplots(figsize=(9, 2.8))
b = b.loc[['ilae', 'engel']]  # rows are drawn bottom-up: Engel on top
labels = [f"{OUTCOMES[o]['name']} ({OUTCOMES[o]['improved_label'].split(' ', 1)[1]} vs "
          f"{OUTCOMES[o]['not_improved_label'].split(' ', 1)[1]})" for o in b.index]
paired_rate_plot(ax, labels, top=('improved', b['drop_free_improved'], b['n_improved'], PAIR_A),
                 bottom=('not improved', b['drop_free_not_improved'], b['n_not_improved'], PAIR_B),
                 header='Drop-free / total', p=b['p_fisher'])
ax.set_xlabel('Drop-free at 6 months (Wilson 95% CI)')
ax.set_title('Drop-free at 6 months, improved vs not improved', loc='left', pad=28)
ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False, handletextpad=0.3)
plt.tight_layout()
save_figure(FOLDER, 'drop_free_by_improvement')
plt.show()

## 5. By seizure type

Among patients with drop seizures and pre-op seizure types recorded: the share drop-free with vs without each core type and LGS, with Fisher's exact test, Holm-corrected across the six. The table adds each group's median reduction.

In [ ]:
t = by_type.assign(difference=by_type['rate_with'] - by_type['rate_without']).sort_values('difference').reset_index(drop=True)
display(pd.DataFrame({
    'Feature': t['feature'].map(pretty),
    'With: drop-free / n': t['drop_free_with'].astype(str) + ' / ' + t['n_with'].astype(str),
    'Without: drop-free / n': t['drop_free_without'].astype(str) + ' / ' + t['n_without'].astype(str),
    'Median reduction (with / without)': [f'{-w:.0f}% / {-wo:.0f}%' for w, wo in t[['median_with', 'median_without']].to_numpy()],
    'p': t['p'].map(fmt_p),
    'Holm p': t['p_holm'].map(fmt_p),
}).style.hide(axis='index'))

fig, ax = plt.subplots(figsize=(9.5, 5))
paired_rate_plot(ax, t['feature'], top=('with', t['drop_free_with'], t['n_with'], PAIR_A),
                 bottom=('without', t['drop_free_without'], t['n_without'], PAIR_B),
                 header='Drop-free / total', p=t['p'], p_holm=t['p_holm'])
ax.set_xlabel('Drop-free at 6 months (Wilson 95% CI)')
ax.set_title(f"Drop-free at 6 months, with vs without each seizure type (n = {int(t.loc[0, ['n_with', 'n_without']].sum())})",
             loc='left', pad=28)
ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False, handletextpad=0.3)
plt.tight_layout()
save_figure(FOLDER, 'drop_free_by_type')
plt.show()

## Reading the results

Checked 2026-10-21 on v2.

- **Most patients with the value became drop-free:** 33 of 52 (63%), and 43 (83%) had at least a 90% reduction. 3 had no change; nobody got worse.
- **Only 52 of 123 patients (42%) have the value**, and it is recorded less often for patients who did not improve (Engel IV 7 of 18, 39%, vs Engel I-III 45 of 84, 54%; ILAE 5-6 9 of 25, 36%, vs ILAE 1-4 41 of 69, 59%). The recorded share is similar across seizure types (40-63%) and LGS (42% vs 54% without). Whether a blank means "no drop seizures before surgery" or "not recorded" is an open question for the data owner; until it is answered, these results describe the patients with a value, not everyone with drop seizures.
- **Drop-seizure reduction follows the outcome class closely** (Engel rho -0.61, ILAE -0.56, both p < 0.001): Engel I 9/9 drop-free, II 10/12, III 13/24, IV 1/7. Part of this is built in, since both scales count drop seizures among the seizures they measure.
- **Engel III is the informative group.** Its patients have "worthwhile improvement" but not seizure freedom, yet 13 of 24 became drop-free and 21 of 24 had at least a 90% reduction. So for many Engel III patients, the seizures that remain are not drop seizures. On ILAE the same holds for class 4 (20 of 32 drop-free, 29 of 32 at least 90%).
- **Improved vs not:** drop-free in 32/45 (71%) improved vs 1/7 (14%) not improved on Engel (Fisher p = 0.007), and 29/41 (71%) vs 2/9 (22%) on ILAE (p = 0.018). The not-improved groups are very small.
- **Seizure type doesn't predict becoming drop-free.** No type differs after Holm (n = 48). Tonic is lowest (12/25, 48%, vs 17/23, 74%, without; p = 0.083, Holm 0.50) and BTC highest (20/29 vs 9/19, p = 0.23). Myoclonic, the stand-out for overall improvement, makes no difference here (10/16 vs 19/32), and neither does LGS (12/20 vs 17/28). So the myoclonic advantage in `1_improvement` is not an advantage in drop seizures specifically.
- **The values are recorded in steps** (100, 95, 90, 75, 50, 25, 0%), so they are treated as ordered categories: drop-free yes / no, medians and ranks.